# TAR mission-planning training (Kaggle)

Trains MaskablePPO for offline mapper/relay mission planning.

## Before you press Save & Run All
1. **Settings → Accelerator → GPU T4 x2** (or GPU P100)
2. **Settings → Internet → On** (needed to `git clone` / `pip install`)
3. Edit `REPO_URL` / `REPO_BRANCH` below if needed
4. Use **Save Version → Save & Run All**, then close the browser

Outputs land in `/kaggle/working` and appear under the notebook **Versions** tab when the run finishes.

A short `TIMESTEPS` proves the pipeline. Real learning needs much larger budgets (200k–1M+) and comparison to baselines.

## 1. Clone the repo

Internet must be enabled. If the GitHub repo is private, upload a zip as a Kaggle Dataset instead and set `USE_DATASET = True`.

In [ ]:
import os
from pathlib import Path

REPO_URL = "https://github.com/AMMistry18/T1_Software_Swarm.git"
REPO_BRANCH = "network"  # change if your code is on another branch
USE_DATASET = False       # True if you uploaded the repo as a Kaggle Dataset
DATASET_DIR = Path("/kaggle/input")  # then point at your dataset folder below

work = Path("/kaggle/working")
os.chdir(work)

if USE_DATASET:
    # Example: /kaggle/input/t1-software-swarm/T1_Software_Swarm
    candidates = list(DATASET_DIR.glob("**/pyproject.toml"))
    if not candidates:
        raise FileNotFoundError("No pyproject.toml under /kaggle/input — check dataset path")
    repo = candidates[0].parent
else:
    repo = work / "T1_Software_Swarm"
    if not repo.exists():
        !git clone --branch {REPO_BRANCH} --single-branch {REPO_URL} {repo}

os.chdir(repo)
print("repo:", repo)
print("files:", sorted(p.name for p in Path('.').iterdir())[:20])

## 2. Install planning dependencies

In [ ]:
import sys
from pathlib import Path
print(sys.version)

# Kaggle currently ships Python 3.13; relax local pin if needed.
pp = Path('pyproject.toml')
text = pp.read_text(encoding='utf-8')
patched = text.replace(
    'requires-python = ">=3.12,<3.13"',
    'requires-python = ">=3.12,<3.14"',
)
if patched != text:
    pp.write_text(patched, encoding='utf-8')
    print('patched pyproject requires-python for Python 3.13')

# Install planning deps only. Do NOT `pip install -e .` with default
# deps: requirements.txt pulls MuJoCo, which is unrelated and heavy here.
!python -m pip -q install -r requirements-planning.txt
!python -m pip -q install -e . --no-deps

src = str(Path('src').resolve())
if src not in sys.path:
    sys.path.insert(0, src)

import tar_swarm
from tar_swarm.mission_planning.scenarios import one_mapper_small
from tar_swarm.mission_planning.train import train_maskable_ppo
print('tar_swarm from', tar_swarm.__file__)
print('imports ok')

import torch
print('cuda available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print(torch.cuda.get_device_name(0))


## 3. Train

- First Kaggle test: `TIMESTEPS = 10_000`
- Real attempt: `200_000` or more
- Checkpoints are written under `/kaggle/working/...` so Version outputs keep them

In [ ]:
from pathlib import Path
from tar_swarm.mission_planning.scenarios import one_mapper_small
from tar_swarm.mission_planning.train import train_maskable_ppo

TIMESTEPS = 10_000      # raise for real training, e.g. 200_000
N_ENVS = 4
SEED = 0
USE_CURRICULUM = True
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

out = Path("/kaggle/working/plan_train_kaggle")
summary = train_maskable_ppo(
    one_mapper_small(),
    out,
    total_timesteps=TIMESTEPS,
    n_envs=N_ENVS,
    seed=SEED,
    use_curriculum=USE_CURRICULUM,
    device=DEVICE,
)
print(summary)
summary

## 4. Evaluate vs baselines

In [ ]:
import json
from tar_swarm.mission_planning.evaluate import evaluate_checkpoint
from tar_swarm.mission_planning.api import plan_mission
from tar_swarm.mission_planning.scenarios import one_mapper_small

ckpt = summary["final_model"]
eval_result = evaluate_checkpoint(
    one_mapper_small(),
    ckpt,
    output_path=out / "eval.json",
)
print(json.dumps(eval_result, indent=2, default=str))

baseline = plan_mission(one_mapper_small(), checkpoint=None, fallback="static")
print("baseline success:", baseline.success, baseline.source)

rl = plan_mission(one_mapper_small(), checkpoint=ckpt)
print("rl success:", rl.success, rl.source)
if rl.metrics:
    keys = [
        "coverage_fraction",
        "disconnected_mapper_s",
        "relay_travel_m",
        "mission_duration_s",
        "success",
    ]
    print({k: rl.metrics.get(k) for k in keys})

## 5. Bundle checkpoint for download

After **Save & Run All** finishes, open the Version → **Output** and download `checkpoint_bundle.zip`.

In [ ]:
import shutil
from pathlib import Path

bundle = Path("/kaggle/working/checkpoint_bundle")
bundle.mkdir(parents=True, exist_ok=True)
ckpt_dir = out / "checkpoints"
for name in ["final_model.zip", "final_model.meta.json"]:
    src = ckpt_dir / name
    if src.exists():
        shutil.copy(src, bundle / name)
        print("copied", src)
    else:
        print("missing", src)

# Also keep best checkpoint if present
best = ckpt_dir / "best" / "best_model.zip"
best_meta = ckpt_dir / "best" / "best_model.meta.json"
if best.exists():
    shutil.copy(best, bundle / "best_model.zip")
if best_meta.exists():
    shutil.copy(best_meta, bundle / "best_model.meta.json")

zip_path = shutil.make_archive("/kaggle/working/checkpoint_bundle", "zip", bundle)
print("bundle ready:", zip_path)
print("working tree:", list(Path("/kaggle/working").glob("*")))

## Resume tip

Upload a previous `final_model.zip` + `.meta.json` as a dataset, then:

```python
summary = train_maskable_ppo(
    one_mapper_small(),
    Path("/kaggle/working/plan_train_kaggle_resume"),
    total_timesteps=200_000,
    n_envs=4,
    resume_path="/kaggle/input/YOUR_DATASET/final_model.zip",
    use_curriculum=True,
    device=DEVICE,
)
```